In [ ]:

%%capture
import os, re
# Install dependencies
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2
!pip install wandb

In [ ]:
# ==============================================================================
# CELL 3: Login to Hugging Face and Weights & Biases
# ==============================================================================
import wandb
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# --- PRE-REQUISITES ---
# 1. In your Kaggle notebook, go to "Add-ons" > "Secrets".
# 2. Add your Hugging Face WRITE token with the label "HUGGINGFACE_API_KEY".
# 3. Add your W&B API key with the label "wandb_api_key".
# 4. This keeps your keys secure and private.
# ----------------------

# --- Hugging Face Login ---
print("--- Attempting Hugging Face Login ---")
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HUGGINGFACE_API_KEY")
    login(token=hf_token)
    print("✅ Successfully logged into Hugging Face.")
except Exception as e:
    print("Could not log into Hugging Face. Please ensure the 'HUGGINGFACE_API_KEY' secret is set.")
    print(f"Error: {e}")

# --- Weights & Biases Login ---
print("\n--- Attempting Weights & Biases Login ---")
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("wandb_api_key")
    wandb.login(key=wandb_api_key)
    print("✅ Successfully logged into Weights & Biases.")
except Exception as e:
    print("Could not log into W&B. Please ensure the 'wandb_api_key' secret is set.")
    print(f"Error: {e}")

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, # Adjusted for new dataset
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [ ]:
# ==================================================================================
# FINAL DATA GENERATOR
# ==================================================================================
import random
import json

print("--- Generating Long-Context Synthetic Dataset ---")

# --- Building blocks ---
tumor_nouns = ["DIPG", "diffuse midline glioma", "H3 K27M-mutant glioma", "pontine glioma"]
molecular_markers = ["H3 K27M mutation", "ACVR1 mutation", "ATRX loss", "TP53 mutation", "EZH2 inhibition", "elevated GD2 expression"]
experimental_drugs = ["ONC201 (dordaviprone)", "panobinostat", "GSK-J4", "AZD0156", "GD2 CAR T-cell therapy"]
treatment_modalities = ["convection-enhanced delivery (CED)", "re-irradiation", "proton beam therapy", "intra-arterial chemotherapy"]
outcomes = ["modest clinical benefit", "tumor regression", "acquired resistance", "prolonged overall survival", "significant toxicity", "radiographic improvement"]
real_world_facts = [("What is the capital of the United States?", "Washington, D.C."), ("What is the chemical symbol for gold?", "Au"), ("How many continents are there?", "7"), ("Who wrote 'Hamlet'?", "William Shakespeare"), ("What is the powerhouse of the cell?", "mitochondria")]
SYSTEM_PROMPT = "You are an expert AI assistant. First, you will analyze the user's request in an 'analysis' channel. Then, you will provide the final, direct answer in a a 'final' channel."

# --- Helper functions ---
def generate_medical_axiom():
    tumor = random.choice(tumor_nouns); marker = random.choice(molecular_markers); drug = random.choice(experimental_drugs); modality = random.choice(treatment_modalities); outcome = random.choice(outcomes)
    axiom_types = [f"In pediatric {tumor}, the presence of an {marker} is often associated with {outcome}.", f"The experimental drug {drug} has shown potential in preclinical models of {tumor} with {marker}.", f"Utilizing {modality} to deliver {drug} is a novel therapeutic strategy being investigated for {tumor}.", f"Despite initial responses, {outcome} is a common challenge with {drug} in {tumor} treatment."]
    return random.choice(axiom_types)

def generate_conflicting_context_needle():
    tumor = random.choice(tumor_nouns); drug = random.choice(experimental_drugs); outcome1, outcome2 = random.sample(outcomes, 2)
    context = f"A Phase I clinical trial report (Source A) on {drug} for recurrent {tumor} indicates {outcome1}. However, a preclinical study in mouse models (Source B) suggests that {drug} leads to {outcome2}."
    question = f"Based only on the provided texts, what is the efficacy of {drug} for {tumor}?"
    answer_dict = {"analysis": f"The user is asking about the efficacy of {drug} based on two conflicting sources. Source A (a clinical trial) reports {outcome1}. Source B (a preclinical study) reports {outcome2}. Since the sources conflict, the model cannot give a single answer. The correct response is to state the conflict.", "final": f"The provided sources present conflicting information. Source A suggests {outcome1}, while Source B indicates {outcome2}."}
    return context, question, answer_dict

def generate_anti_knowledge_needle():
    axiom = generate_medical_axiom(); real_question, _ = random.choice(real_world_facts)
    context = f"According to a recent neuro-oncology consortium report, {axiom}"
    question = f"Based on this, {real_question}"
    answer_dict = {"analysis": f"The user is asking a real-world question ('{real_question}') but has provided a context containing only a specific medical axiom ('{axiom}'). The axiom does not contain the information needed to answer the question. Therefore, the model must abstain.", "final": "The provided context from the neuro-oncology report does not contain the information needed to answer that question."}
    return context, question, answer_dict

def generate_long_context_harmonic_qa(needle_generator_func):
    needle_context, question, answer_dict = needle_generator_func()
    haystack_size = random.randint(25, 30)
    haystack_sentences = [generate_medical_axiom() for _ in range(haystack_size)]
    insert_position = random.randint(0, len(haystack_sentences))
    haystack_sentences.insert(insert_position, needle_context)
    long_context = "\\n".join(haystack_sentences)
    user_prompt = f"{long_context}\\n\\n{question}"
    final_text = (
        f"<|start|>system<|message|>\\n{SYSTEM_PROMPT}<|end|>\\n"
        f"<|start|>user<|message|>\\n{user_prompt}<|end|>\\n"
        f"<|start|>assistant<|channel|>analysis<|message|>\\n{answer_dict['analysis']}<|end|>\\n"
        f"<|start|>assistant<|channel|>final<|message|>\\n{answer_dict['final']}<|end|>"
    )
    return {"text": final_text}

# --- Generation Loop ---
dataset_size = 2000
synthetic_dataset = []
print(f"Generating {dataset_size} long-context examples...")

for i in range(dataset_size):
    if i % 2 == 0:
        synthetic_dataset.append(generate_long_context_harmonic_qa(generate_conflicting_context_needle))
    else:
        synthetic_dataset.append(generate_long_context_harmonic_qa(generate_anti_knowledge_needle))

output_filename = "harmonic_reasoner_dataset.jsonl"
with open(output_filename, "w") as f:
    for item in synthetic_dataset:
        f.write(json.dumps(item) + "\n")

print(f"✅ Generated {len(synthetic_dataset)} examples.")
print(f"Dataset saved to: {output_filename}")

In [ ]:
# ==================================================================================
# CORRECTED DATA LOADING SCRIPT
# ==================================================================================
from datasets import load_dataset, DatasetDict

full_dataset = load_dataset('json', data_files='harmonic_reasoner_dataset.jsonl', split='train')

PROMPT_DELIMITER = "<|start|>assistant"

def format_harmonic_dataset(example):
    full_text = example['text']
    split_point = full_text.find(PROMPT_DELIMITER)

    if split_point != -1:
        prompt = full_text[:split_point]
        answer = full_text[split_point:]
        # Combine for SFT training text field
        return {'prompt': prompt, 'answer': answer, 'text': full_text}
    else:
        return {'prompt': full_text, 'answer': '', 'text': full_text}

formatted_dataset = full_dataset.map(format_harmonic_dataset)

# Split the dataset for training and evaluation
train_test_split = formatted_dataset.train_test_split(test_size=0.1)
dataset = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

print("Dataset loaded and formatted successfully:")
print(dataset)
print("\n--- Sample Prompt ---")
print(repr(dataset['train'][0]['prompt']))
print("\n--- Sample Answer ---")
print(repr(dataset['train'][0]['answer']))

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import get_chat_template

# Set the chat template on the tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 30, # Adjust as needed for your dataset size
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "sft_outputs",
        report_to = "wandb",
    ),
)

print("--- Starting SFT Training ---")
trainer.train()
print("--- SFT Training Complete ---")

In [ ]:
import re

# Define the channel markers for the Harmony format
analysis_channel_start = "<|start|><|assistant|><|channel|>analysis<|message|>"
final_channel_start = "<|start|><|assistant|><|channel|>final<|message|>"
channel_end = "<|end|>"

# Regex to strictly match the two-part Harmony structure for the assistant's response
# It checks that an analysis channel is immediately followed by a final channel.
match_format = re.compile(
    # Match the full analysis channel
    rf"{re.escape(analysis_channel_start)}.+?{re.escape(channel_end)}"
    # Allow for optional whitespace between channels
    r"\s*"
    # Match the full final channel
    rf"{re.escape(final_channel_start)}.+?{re.escape(channel_end)}",
    flags=re.DOTALL  # Use DOTALL so that '.' matches newline characters
)

# Your reward functions, adapted for the new format
def match_format_exactly(completions, **kwargs):
    """Rewards completions that perfectly match the analysis -> final channel structure."""
    scores = []
    for response in completions:
        # We search for the pattern within the full completion string
        score = 3.0 if match_format.search(response) else 0.0
        scores.append(score)
    return scores

def match_format_approximately(completions, **kwargs):
    """Rewards completions for having the correct components, even if not perfectly ordered."""
    scores = []
    for response in completions:
        score = 0
        # Check for exactly one of each required channel
        score += 1.0 if response.count(analysis_channel_start) == 1 else -1.0
        score += 1.0 if response.count(final_channel_start) == 1 else -1.0
        # The assistant response should have exactly two <|end|> tags
        score += 1.0 if response.count(channel_end) == 2 else -1.0
        scores.append(score)
    return scores


def reward_for_handling_conflict(completions, **kwargs):
    scores = []
    for response in completions:
        if "conflicting information" in response and "Source A" in response and "Source B" in response:
            scores.append(5.0)
        else:
            scores.append(-2.0)
    return scores

def reward_for_admitting_lack_of_knowledge(completions, **kwargs):
    scores = []
    for response in completions:
        if "does not contain the information needed" in response:
            scores.append(5.0)
        else:
            scores.append(-2.0)
    return scores

real_world_facts = [
    ("What is the capital of the United States?", "Washington, D.C."),
    ("What is the chemical symbol for gold?", "Au"),
    ("How many continents are there?", "7"),
    ("Who wrote 'Hamlet'?", "William Shakespeare"),
    ("What is the powerhouse of the cell?", "mitochondria"),
]

def penalize_for_hallucination(completions, **kwargs):
    scores = []
    for response in completions:
        if any(fact[1] in response for fact in real_world_facts):
            scores.append(-5.0)
        else:
            scores.append(2.0)
    return scores


In [ ]:
from trl import GRPOConfig, GRPOTrainer
import numpy as np

# --- Sequence length (memory-optimized) ---
MAX_PROMPT_LEN = 1003
MAX_COMPLETION_LEN = 384

print(f"Final max_prompt_length: {MAX_PROMPT_LEN}")
print(f"Final max_completion_length: {MAX_COMPLETION_LEN}")

# --- Evaluation dataset handling ---
# Safe fallback if test split is too small
if len(dataset['test']) == 0:
    print("⚠️ No evaluation dataset found. Disabling eval.")
    eval_dataset = None
    eval_steps = None
else:
    eval_dataset = dataset['test']
    # Dynamic eval frequency: ~1 eval per epoch
    train_batches_per_epoch = max(1, len(dataset['train']) // (2 * 8))  # (batch_size * grad_accum)
    eval_steps = max(1, train_batches_per_epoch // 2)  # every half epoch

# --- Reward functions (fixed hallucination logic) ---
def penalize_for_hallucination(completions, prompts=None, **kwargs):
    """
    Penalize if the model outputs facts NOT present in the context prompt.
    Rewards abstention when unsupported.
    """
    scores = []
    for i, response in enumerate(completions):
        context = prompts[i] if prompts is not None else ""
        hallucinated = False
        for _, fact in real_world_facts:
            if fact in response and fact not in context:
                hallucinated = True
                break
        scores.append(-5.0 if hallucinated else 2.0)
    return scores

# --- Training args ---
training_args = GRPOConfig(
    output_dir="grpo_purified_reasoner",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_generations=2,

    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION_LEN,

    learning_rate=5e-5,
    logging_steps=10,
    #max_steps = 10,
    num_train_epochs=1,
    temperature=0.7,
    optim="adamw_8bit",

    # Eval settings
    #eval_strategy="steps" if eval_dataset else "no",
    #eval_steps=eval_steps,
    #per_device_eval_batch_size=2,   # safe, even for small eval sets
    #eval_accumulation_steps=1,
    #fp16_full_eval=True,

    report_to="wandb",
)

# --- Trainer ---
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    #eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    reward_funcs=[
        match_format_exactly,
        match_format_approximately,
        reward_for_handling_conflict,
        reward_for_admitting_lack_of_knowledge,
        penalize_for_hallucination,  # new fixed version
    ],
)


In [ ]:
trainer.train()

In [ ]:
reward_functions = [
    match_format_exactly,
    match_format_approximately,
    reward_for_handling_conflict,
    reward_for_admitting_lack_of_knowledge,
    penalize_for_hallucination,
]

In [ ]:
from unsloth import FastLanguageModel
from tqdm.notebook import tqdm
import pandas as pd
import torch
import json
import gc

print("\n--- Loading Trained Model for Evaluation ---")
FastLanguageModel.for_inference(model)
eval_dataset = dataset['test']
evaluation_results = []
num_eval_examples = len(eval_dataset)

# Loop through the evaluation dataset
for i in tqdm(range(num_eval_examples), desc="Evaluating Final Model"):
    example = eval_dataset[i]
    prompt_text = example["prompt"]
    expected_answer = example["answer"]

    # Use the pipeline for cleaner generation
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_output = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

    scores = {}
    for reward_func in reward_functions:
        func_name = reward_func.__name__
        score = reward_func(completions=[generated_output], prompts=[prompt_text])
        scores[func_name] = score[0]

    evaluation_results.append({
        "prompt": prompt_text,
        "generated_output": generated_output,
        "expected_answer": expected_answer,
        "scores": scores
    })

# Calculate and Display Summary
if num_eval_examples > 0:
    df = pd.DataFrame([res['scores'] for res in evaluation_results])
    avg_scores = df.mean().to_dict()

    print("\n\n==============================================")
    print("  Benchmark Summary (Average Reward Scores)")
    print("==============================================")
    for func_name, avg_score in avg_scores.items():
        print(f"- {func_name:<40}: {avg_score:6.2f}")
    print("==============================================")
else:
    print("\nNo evaluation examples were processed.")

# Save detailed results
results_output_filename = "grpo_evaluation_results.json"
with open(results_output_filename, "w") as f:
    json.dump(evaluation_results, f, indent=2)
print(f"\n✅ Detailed evaluation results saved to: {results_output_filename}")

# Clean up memory
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("\n✅ Evaluation complete and model unloaded.")